In [11]:
import torch
import numpy as np

In [13]:
# Weapons detection (YOLOv8)
from ultralytics import YOLO
weapon_model = YOLO(r"yolov8n.pt")

In [14]:
# Violence detection (SlowFast)
import torch.nn as nn
violence_model = torch.hub.load('facebookresearch/pytorchvideo', 'slowfast_r50', pretrained=False)
violence_model.blocks[-1].proj = nn.Linear(violence_model.blocks[-1].proj.in_features, 3)
violence_model.load_state_dict(torch.load(r"violence-detection-slowfast-model.pth"))
violence_model.eval()

Using cache found in C:\Users\risha/.cache\torch\hub\facebookresearch_pytorchvideo_main


Net(
  (blocks): ModuleList(
    (0): MultiPathWayWithFuse(
      (multipathway_blocks): ModuleList(
        (0): ResNetBasicStem(
          (conv): Conv3d(3, 64, kernel_size=(1, 7, 7), stride=(1, 2, 2), padding=(0, 3, 3), bias=False)
          (norm): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (activation): ReLU()
          (pool): MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=[0, 1, 1], dilation=1, ceil_mode=False)
        )
        (1): ResNetBasicStem(
          (conv): Conv3d(3, 8, kernel_size=(5, 7, 7), stride=(1, 2, 2), padding=(2, 3, 3), bias=False)
          (norm): BatchNorm3d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (activation): ReLU()
          (pool): MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=[0, 1, 1], dilation=1, ceil_mode=False)
        )
      )
      (multipathway_fusion): FuseFastToSlow(
        (conv_fast_to_slow): Conv3d(8, 16, kernel_size=(7, 1, 1), st

In [16]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim=2048, hidden_dim=512):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim//2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

# Anomaly detection (Autoencoder)
anomaly_model = Autoencoder()
anomaly_model.load_state_dict(torch.load("autoencoder_ucfcrime_epoch10.pth"))
anomaly_model.eval()

Autoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=2048, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU()
  )
  (decoder): Sequential(
    (0): Linear(in_features=256, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=2048, bias=True)
  )
)

In [21]:
def fusion_rule(weapon_conf, violence_conf, anomaly_score,
                weapon_thresh=0.5, violence_thresh=0.5, anomaly_thresh=0.5):
    """
    Rule-based fusion strategy:
    1. If weapon detected above threshold → immediate threat.
    2. Else if violence detected above threshold → threat.
    3. Else if anomaly score above threshold → possible threat.
    4. Otherwise → safe.
    """
    if weapon_conf >= weapon_thresh:
        return "Threat detected! : Weapon"
    elif violence_conf >= violence_thresh:
        return "Threat detected! : Violence"
    elif anomaly_score >= anomaly_thresh:
        return "Threat detected! : Anomaly"
    else:
        return "Safe"

In [25]:
# Testing dummy outputs (to be replaced with real inference)
weapon_conf = 0.49   # YOLO confidence for weapon
violence_conf = 0.55 # SlowFast violence probability
anomaly_score = 0.61 # Autoencoder reconstruction error normalized

decision = fusion_rule(weapon_conf, violence_conf, anomaly_score)
print("Fusion Decision:", decision)

Fusion Decision: Threat detected! : Violence


In [26]:
fusion_log = {
    "weapon_conf": weapon_conf,
    "violence_conf": violence_conf,
    "anomaly_score": anomaly_score,
    "fusion_decision": decision
}
print(fusion_log)

{'weapon_conf': 0.49, 'violence_conf': 0.55, 'anomaly_score': 0.61, 'fusion_decision': 'Threat detected! : Violence'}
